[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/doubled_algebra/doubled_algebra.ipynb)

# The Doubled Algebra

This example constructs every linear map on the exterior algebra of a vector space as left multiplication by a single multivector in a split algebra of twice the dimension. Pairing each basis vector of the original space with an opposite-square basis vector produces two null directions whose geometric products insert a basis factor like the exterior wedge and remove one like contraction. Because both operations and all of their combinations live inside one geometric product, applying linear maps one after another reduces to multiplying the multivectors that represent them.

From a positive-square basis vector `x` and an opposite-square basis vector `u`, the null combinations `(x + u) / 2` and `(x - u) / 2` insert and remove `x` on exterior states embedded into a minimal left ideal by `State * projector`. Left multiplication then acts as a linear map `State <- State`, mixed products `(x + u) * (y - v) / 4` act on blades as derivations whose exponential carries areas and volumes along with vectors, and tracing any `State <- State` map against the represented blades recovers its encoding multivector through `Full.scalar_product(Full)`.

In fermionic Fock-space notation and the exterior-operator notation of Gualtieri's [Generalized complex geometry, §1.3](https://www.math.toronto.edu/mgualt/gcg/gcg.pdf#page=5), exterior multiplication $\epsilon(e_i)\,\omega = e_i \wedge \omega$ and interior contraction $\iota(e_i^*)\,\omega = e_i^* \mathbin{\lrcorner} \omega$ are introduced as separate operator families on $\Lambda \mathbb{R}^n$ satisfying the canonical anticommutation brackets $\{a_i, a_j^\dagger\} = \delta_{ij}$ and $\{a_i, a_j\} = \{a_i^\dagger, a_j^\dagger\} = 0$ above a vacuum $|0\rangle$, a matrix $A^i{}_j$ lifts to $k$-blades separately as the derivation $\sum_{r=1}^k v_1 \wedge \dots \wedge A v_r \wedge \dots \wedge v_k$ and the compound outermorphism $\Lambda^k A\,(v_1 \wedge \dots \wedge v_k) = A v_1 \wedge \dots \wedge A v_k$, and a general operator in $\mathrm{End}(\Lambda \mathbb{R}^n) \cong \mathrm{Cl}(n, n)$ expands as a normal-ordered polynomial in $a_{i_1}^\dagger \cdots a_{i_p}^\dagger a_{j_1} \cdots a_{j_q}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
from functools import reduce
from operator import mul

import numpy as np

from numga import Algebra, NumpyContext, stack
from examples.math.doubled_algebra import render, scenarios
from examples.util.animation import save_figure

np.set_printoptions(precision=3, suppress=True)

# Two positive-square and two negative-square basis vectors give Cl(2, 2), with sixteen blades.
ga = Algebra("x+y+u-v-")
context = NumpyContext(ga)
mv = context.multivector
Full = ga.gatype.full()                                     # Full+
State = Full.derive.within(Algebra("x+y+"))                 # Full+
Vector = ga.gatype.vector().derive.within(Algebra("x+y+"))  # Vector+
StateMap = ga.gatype((State, State))                        # State+ <- State+
n_state_components = len(State.output_subspace)

## Adding and removing a basis vector

Wedging a basis vector into a blade inserts it as a factor, while contraction removes it with the sign set by its position in the blade. Both operations square to zero because neither can insert or remove the same basis factor twice. Combining each positive-square basis vector with an opposite-square basis vector yields two null vectors whose geometric products obey those exact rules, while their anticommutator recovers the metric pairing of the original basis.

In [ ]:
basis = stack([mv.x, mv.y])                                 # [2] Vector+
negative_basis = stack([mv.u, mv.v])                        # [2] Vector+
creators = (basis + negative_basis) / 2                     # [2] Vector+
removers = (basis - negative_basis) / 2                     # [2] Vector+

# Each family consists of null vectors, while the matching pair has unit anticommutator.
create_x, create_y = creators                               # [] Vector+, Vector+
remove_x, remove_y = removers                               # [] Vector+, Vector+
pairing_x = 2 * remove_x.anticommutator(create_x)           # [] Scalar+

print("Add x")
print(create_x.coefficient_table())
print("Remove x")
print(remove_x.coefficient_table())
print("Anticommutator of x pair")
print(pairing_x.coefficient_table())

## Left multiplication as a map on states

Every remover annihilates the empty state at the bottom of the exterior algebra. Multiplying that empty state on the left by creators generates all four exterior basis blades inside a minimal left ideal that stays closed under left multiplication by the entire doubled algebra. Embedding `State` into the ideal with `State * projector`, multiplying on the left by any multivector, and rescaling back onto `State` turns left multiplication directly into a linear map `State <- State`.

In [ ]:
# Removing either basis vector from this state gives zero; the two empty-state projectors commute.
projector = reduce(mul, removers * creators)                # [] Even+
embedder = State * projector                                # [] Full+ <- State+
# Each empty-state projector contributes a factor of one half that reading out rescales away.
state_readout = (n_state_components * Full).cast(State)     # [] State+ <- Full+
representations = state_readout(Full * embedder)            # [] State+ <- (Full+, State+)

insertion = representations(create_x)                       # [] State+ <- State+
removal = representations(remove_x)                         # [] State+ <- State+

print("Add x (State <- State)")
print(insertion.coefficient_table())
print("Remove x (State <- State)")
print(removal.coefficient_table())

In [ ]:
state = 1 + 2 * mv.x - mv.y + 3 * mv.xy                     # [] State+

# Summing create_x and remove_x recovers x, whose geometric product both wedges and contracts.
left_multiplication = representations(create_x + remove_x)  # [] State+ <- State+
product = left_multiplication(state)                        # [] State+

print("State")
print(state.coefficient_table())
print("Left multiplication by x (State <- State)")
print(left_multiplication.coefficient_table())
print("Product")
print(product.coefficient_table())

## Shear from one mixed product

Multiplying a remover for `y` on the left by a creator for `x` forms a dyad that maps `y` to `x` and annihilates `x`. On a higher-grade blade, that mixed product replaces each factor in turn according to the derivation product rule, so exponentiating it shears vectors while carrying their exterior wedges along as an outermorphism. Shearing a square therefore tilts it into a parallelogram of the same oriented area.

In [ ]:
shear_amount = scenarios.shear_amount
extent, n_lines = scenarios.extent, scenarios.n_lines
boundary = mv(Vector, [[-1, -1], [1, -1], [1, 1], [-1, 1]])  # [4] Vector+
grid = scenarios.grid_segments(basis, extent, n_lines)       # [2 * n_lines, 2] Vector+

In [ ]:
# Removing y and adding x sends y to x and extends to blades by the derivation product rule.
shear_generator = creators[0] * removers[1]                 # [] Even+
# Because shear_generator squares to zero, its exponential terminates after the linear term.
transformation = representations((shear_generator * shear_amount).exp())  # [] State+ <- State+
# The shear preserves grades, so its restriction to vectors maps vectors to vectors.
vector_shear = transformation(Vector).cast(Vector)          # [] Vector+ <- Vector+
grid_image = vector_shear(grid)                             # [2 * n_lines, 2] Vector+
boundary_image = vector_shear(boundary)                     # [4] Vector+
basis_image = vector_shear(basis)                           # [2] Vector+
area_image = transformation(mv.xy)                          # [] State+

print("Shear generator (State <- State)")
print(representations(shear_generator).coefficient_table())
print("Image of the unit area")
print(area_image.coefficient_table())
save_figure(render.shear_figure(grid, grid_image, boundary, boundary_image, basis_image), "doubled_algebra_shear");

The grey grid and dashed square show the reference shape, while the blue grid and orange boundary show its sheared image. The teal and purple arrows trace the images of `x` and `y`, holding `x` fixed while `y` tilts toward it.

Plotting each vector by pairing with the duals of `x` and `y` reads the shear on `Vector` directly as motion in the plane, which departs from the plane-based convention where points are antivectors and planar vectors represent lines through the origin. Because the exterior states here are built by wedging vectors, grade-three antivectors of the four-dimensional doubled algebra mix in the negative-square basis `u` and `v` and leave no pure planar point subspace. Taking complements within the `(x, y)` plane instead moves points the opposite way under the same exponential, keeping the `y` axis fixed while tilting the `x` axis away from `y`; whether that tension reflects a limit of the orientation paradigm or of a vector-based spinor construction remains an open question.

## Products beyond vector maps

Removing and reinserting a basis vector retains only the blades that contain that factor, so summing `creators * removers` across the basis multiplies each grade by its factor count. Products of two creators or two removers instead change grade by two, inserting or stripping a pair of basis factors at once. Adding `create_x * create_y` to `remove_y * remove_x` swaps the scalar and area components while annihilating every vector, producing a map on states whose action vanishes completely on `Vector`.

In [ ]:
# Each diagonal product keeps the blades containing its basis vector, so their sum counts grade.
grade_count = (creators * removers).sum(axis=0)             # [] Even+
# Adding both basis vectors or removing them in reverse order swaps the scalar and area blades.
exchange = creators[0] * creators[1] + removers[1] * removers[0]  # [] Even+
exchanged_state = representations(exchange)(state)          # [] State+

print("Count basis vectors (State <- State)")
print(representations(grade_count).coefficient_table())
print("Exchange scalar and area (State <- State)")
print(representations(exchange).coefficient_table())
print("Exchanged state")
print(exchanged_state.coefficient_table())

## Every unary map as a multivector

Tracing any prescribed linear map on four state components against the representations of all sixteen blades and solving with `Full.scalar_product(Full)` recovers the unique multivector whose left action reproduces that map. Composing two linear maps on `State` then matches the geometric product of their encoding multivectors.

In [ ]:
def map_multivector(state_map: StateMap) -> Full:
    """Trace pairing against all sixteen represented blades recovers the multivector (`[] Full+`) whose left action equals `state_map`."""
    pairing = Full.scalar_product(Full)                     # [] Scalar+ <- (Full+, Full+)
    trace_rhs = representations(Full, state_map).trace(0, 2) / n_state_components  # [] Full+
    return pairing.solve(trace_rhs)


probe_vector = 2 * mv.x + mv.y                              # [] Vector+
target_state = 1 + mv.x - mv.xy                             # [] State+
coupling = 0.3

# Combine a rank-one scalar readout into mixed grades with exterior insertion of y.
state_map = target_state * probe_vector.scalar_product(State) + coupling * (mv.y ^ State)  # [] State+ <- State+
encoding = map_multivector(state_map)                       # [] Full+
represented_map = representations(encoding)                 # [] State+ <- State+

print("Prescribed map (State <- State)")
print(state_map.coefficient_table())
print("Encoding multivector")
print(encoding.coefficient_table())
print("Represented map (State <- State)")
print(represented_map.coefficient_table())

## Same construction in four dimensions

Pairing four positive-square basis vectors with four negative-square basis vectors forms a 256-blade split algebra acting on all 16 exterior components of four-dimensional space by the identical ideal construction. Even in 256 dimensions, a single mixed product `(x + u) * (y - v) / 4` carries only four nonzero blade coefficients, and its left action on the trivector `yzt` replaces `y` with `x` while leaving `z` and `t` untouched.

In [ ]:
ga4 = Algebra("x+y+z+t+u-v-w-s-")
context4 = NumpyContext(ga4)
mv4 = context4.multivector
Full4 = ga4.gatype.full()                                   # Full+
State4 = Full4.derive.within(Algebra("x+y+z+t+"))           # Full+
n_state_components4 = len(State4.output_subspace)

basis4 = stack([mv4.x, mv4.y, mv4.z, mv4.t])                # [4] Vector+
negative_basis4 = stack([mv4.u, mv4.v, mv4.w, mv4.s])       # [4] Vector+
creators4 = (basis4 + negative_basis4) / 2                  # [4] Vector+
removers4 = (basis4 - negative_basis4) / 2                  # [4] Vector+

projector4 = reduce(mul, removers4 * creators4)             # [] Even+
embedder4 = State4 * projector4                             # [] Full+ <- State+
state_readout4 = (n_state_components4 * Full4).cast(State4)  # [] State+ <- Full+
representations4 = state_readout4(Full4 * embedder4)        # [] State+ <- (Full+, State+)

# The mixed product still has four nonzero blades and acts on every factor of a trivector.
generator4 = creators4[0] * removers4[1]                    # [] Even+
trivector_state = mv4.yzt                                   # [] State+
trivector_image = representations4(generator4)(trivector_state)  # [] State+

print("Generator")
print(generator4.coefficient_table())
print("Trivector state")
print(trivector_state.coefficient_table())
print("Image")
print(trivector_image.coefficient_table())

In [ ]:
# checks
wedge_x = mv.x ^ State                                      # [] State+ <- State+
contraction_x = mv.x.left_contraction(State)                # [] State+ <- State+
assert (creators.squared().abs() < 1e-12).all()
assert (removers.squared().abs() < 1e-12).all()
assert (pairing_x - 1).abs() < 1e-12
assert (insertion - wedge_x).svdvals().max() < 1e-12
assert (removal - contraction_x).svdvals().max() < 1e-12
assert (product - mv.x * state).scalar_norm_squared() < 1e-12
assert (area_image - mv.xy).scalar_norm_squared() < 1e-12
assert (basis_image[0] - mv.x).scalar_norm_squared() < 1e-12
assert (basis_image[1] - (mv.y + shear_amount * mv.x)).scalar_norm_squared() < 1e-12
assert (exchanged_state - (3 + mv.xy)).scalar_norm_squared() < 1e-12
assert (representations(shear_generator) - insertion(representations(remove_y))).svdvals().max() < 1e-12
assert (represented_map - state_map).svdvals().max() < 1e-12
assert (trivector_image - mv4.xzt).scalar_norm_squared() < 1e-12